In [1]:
import json
import pandas as pd
from rank_bm25 import BM25Okapi

In [3]:
with open("catalog.json") as f:
    products = list(json.load(f).values())

In [7]:
def preprocess(product):
    text = " ".join([
        product.get("productDisplayname", ""),
        product.get("category", ""),
        product.get("sub_category", ""),
        product.get("sub_genre", ""),
        product.get("product_description", ""),
        product.get("search_keywords", ""),
        product.get("all_attributes_value", "")
    ])
    return text.lower().split()

corpus = [preprocess(p) for p in products]
bm25 = BM25Okapi(corpus)

In [11]:
def search_raw(query, top_n=5):
    tokens = query.lower().split()
    scores = bm25.get_scores(tokens)

    ranked = sorted(
        zip(products, scores),
        key=lambda x: x[1],
        reverse=True
    )

    return [p["productDisplayname"] for p, _ in ranked[:top_n]]

In [13]:
def ner_model(query):
    # ⚠️ Replace this with your real NER output
    return {
        "intent": "product_search",
        "entities": [
            {
                "id": 1,
                "product_name": "phone",
                "is_main_product": True,
                "connections": "5g"
            }
        ]
    }

def build_query(entity):
    parts = []

    if entity.get("product_name"):
        parts.extend([entity["product_name"]] * 3)

    for key, value in entity.items():
        if key not in ["id", "is_main_product", "product_name"] and value:
            parts.append(value)

    return " ".join(parts)
    
def filter_category(products, entity):
    name = entity.get("product_name", "")
    tokens = name.lower().split()

    filtered = []

    for p in products:
        category_text = " ".join([
            p.get("category", ""),
            p.get("sub_category", ""),
            p.get("sub_genre", "")
        ]).lower()

        # if any token matches category fields
        if any(token in category_text for token in tokens):
            filtered.append(p)

    # fallback: if nothing matched, return all
    return filtered if filtered else products
    
def boost_attributes(results, entity):
    boosted = []

    for r in results:
        product = r["product"]
        score = r["score"]

        attrs = product.get("all_attributes_value", "").lower()

        for key, value in entity.items():
            if key not in ["id", "is_main_product", "product_name"] and value:
                if value.lower() in attrs:
                    score *= 1.5

        boosted.append({"product": product, "score": score})

    return boosted

def search_ner(query, top_n=5):
    ner_output = ner_model(query)
    entities = ner_output["entities"]

    main = next(e for e in entities if e.get("is_main_product"))

    query_built = build_query(main)

    filtered = filter_category(products, main)

    tokens = query_built.lower().split()
    scores = bm25.get_scores(tokens)

    results = [
        {"product": p, "score": s}
        for p, s in zip(products, scores)
        if p in filtered
    ]

    results = boost_attributes(results, main)

    results = sorted(results, key=lambda x: x["score"], reverse=True)

    return [r["product"]["productDisplayname"] for r in results[:top_n]]

In [15]:
def run_batch(input_csv, output_csv):
    df = pd.read_csv(input_csv)

    output_rows = []

    for _, row in df.iterrows():
        query = row["query"]

        raw_results = search_raw(query)
        ner_results = search_ner(query)

        output_rows.append({
            "query": query,
            "raw_top5": ", ".join(raw_results),
            "ner_top5": ", ".join(ner_results)
        })

    out_df = pd.DataFrame(output_rows)
    out_df.to_csv(output_csv, index=False)

    print(f"Saved results to {output_csv}")

In [17]:
run_batch("queries.csv", "results.csv")

Saved results to results.csv
